# Evidence-first RCA with Elastic Agent BuilderRun the cells in order. They generate the incident, register the three ES|QL tools and the agent,run the investigation, and delete everything again at the end.Before you start: Elasticsearch and Kibana 9.5, a generative AI connector configured for AgentBuilder, and a `.env` copied from `.env.example`. The API key needs write access to the OTel datastreams and access to the Kibana Agent Builder APIs.

In [ ]:
%pip install -q -r requirements.txt

## 1. Connect

In [ ]:
import jsonimport osimport randomfrom datetime import datetime, timedelta, timezoneimport requestsfrom dotenv import load_dotenvfrom elasticsearch import Elasticsearch, helpersload_dotenv()ES_URL = os.environ["ELASTICSEARCH_URL"]API_KEY = os.environ["ELASTICSEARCH_API_KEY"]KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")SPACE = os.getenv("KIBANA_SPACE", "default")BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"headers = {    "Authorization": f"ApiKey {API_KEY}",    "kbn-xsrf": "true",    "Content-Type": "application/json",}es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=60)LOGS = "logs-generic.otel-default"TRACES = "traces-generic.otel-default"print(f"elasticsearch {es.info()['version']['number']} | space={SPACE}")

## 2. Set the incident`checkout-api` calls `pricing-api`, which calls `fx-rates`. `search-api` sits outside that path.`fx-rates` breaks at 09:55:33.991; the deploy and the `search-api` timeouts happen in the sameminute and cause nothing.The window is 09:50 to 10:02 today, so the data always lands inside a recent search range.

In [ ]:
random.seed(100)DAY = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)WINDOW_START = DAY + timedelta(hours=9, minutes=50)WINDOW_END = DAY + timedelta(hours=10, minutes=2)DEPLOY_AT = DAY + timedelta(hours=9, minutes=55)                       # the release that did not do itFX_BREAKS_AT = DAY + timedelta(hours=9, minutes=55, seconds=33, milliseconds=991)SEARCH_BREAKS_AT = DAY + timedelta(hours=9, minutes=55, seconds=34, milliseconds=111)OLD_VERSION, NEW_VERSION = "2026.07.25.3", "2026.07.26.2"FX_VERSION, PRICING_VERSION, SEARCH_VERSION = "2026.07.19.1", "2026.07.22.4", "2026.07.24.1"CHECKOUT_REQUESTS = 7374        # split across the two checkout-api versionsSEARCH_REQUESTS = 3637FX_REFUSAL_RATE = 0.517         # after FX_BREAKS_AT, which lands near a third of the whole windowSEARCH_TIMEOUT_RATE = 0.704     # after SEARCH_BREAKS_ATSNAPSHOT_AGE_AT_BREAK = 1127    # seconds, already past max_age when the window opensMAX_AGE = 900DEMO_SERVICES = ["checkout-api", "pricing-api", "fx-rates", "search-api"]print(f"window {WINDOW_START.isoformat().replace('+00:00', 'Z')}"      f" to {WINDOW_END.isoformat().replace('+00:00', 'Z')}")

## 3. Build the documentsTwo shapes, matching what the native OTLP endpoint writes: server spans with`attributes.http.status_code` and `resource.attributes.service.version`, and error logs with`trace_id`, `attributes.error.kind` and `attributes.upstream.service`.

In [ ]:
def ts(moment):    return moment.isoformat().replace("+00:00", "Z")def hexid(n):    return "".join(random.choice("0123456789abcdef") for _ in range(n))def span(moment, service, version, name, status_code, trace_id, duration_ms):    return {        "_index": TRACES,        "_op_type": "create",        "_source": {            "@timestamp": ts(moment),            "trace_id": trace_id,            "span_id": hexid(16),            "name": name,            "kind": "Server",            "duration": int(duration_ms * 1_000_000),            "status": {"code": "Error" if status_code >= 500 else "Ok"},            "attributes": {                "http.request.method": "POST" if service != "search-api" else "GET",                "http.route": name,                "http.status_code": status_code,            },            "resource": {                "attributes": {                    "service.name": service,                    "service.version": version,                    "deployment.environment": "production",                    "host.name": "otel-demo-01",                }            },        },    }def log(moment, service, version, severity, body, trace_id, error_kind=None, upstream=None):    attributes = {}    if error_kind:        attributes["error.kind"] = error_kind    if upstream:        attributes["upstream.service"] = upstream    return {        "_index": LOGS,        "_op_type": "create",        "_source": {            "@timestamp": ts(moment),            "severity_text": severity,            "severity_number": 17 if severity == "ERROR" else 9,            "trace_id": trace_id,            "span_id": hexid(16),            "body": {"text": body},            "attributes": attributes,            "resource": {                "attributes": {                    "service.name": service,                    "service.version": version,                    "deployment.environment": "production",                    "host.name": "otel-demo-01",                }            },        },    }

In [ ]:
docs = []window_seconds = (WINDOW_END - WINDOW_START).total_seconds()# --- the checkout call path: checkout-api -> pricing-api -> fx-rates ---------for i in range(CHECKOUT_REQUESTS):    started = WINDOW_START + timedelta(seconds=random.uniform(0, window_seconds))    trace_id = hexid(32)    # both versions serve traffic across the whole window, which is what makes the    # version split readable at all    checkout_version = NEW_VERSION if i % 2 == 0 else OLD_VERSION    broken = started >= FX_BREAKS_AT and random.random() < FX_REFUSAL_RATE    fx_status = 503 if broken else 200    upstream_status = 500 if broken else 200    docs.append(span(started + timedelta(milliseconds=2), "fx-rates", FX_VERSION,                     "GET /rates/quote", fx_status, trace_id,                     random.uniform(4, 18)))    docs.append(span(started + timedelta(milliseconds=1), "pricing-api", PRICING_VERSION,                     "POST /price", upstream_status, trace_id,                     random.uniform(20, 60)))    docs.append(span(started, "checkout-api", checkout_version,                     "POST /checkout", upstream_status, trace_id,                     random.uniform(45, 140)))    if broken:        age = SNAPSHOT_AGE_AT_BREAK + int((started - FX_BREAKS_AT).total_seconds())        docs.append(log(            started + timedelta(milliseconds=2), "fx-rates", FX_VERSION, "ERROR",            f"StaleQuoteError: fx snapshot age {age}s exceeds max_age {MAX_AGE}s (provider=ecb-eod)",            trace_id, error_kind="StaleQuoteError"))        docs.append(log(            started + timedelta(milliseconds=3), "pricing-api", PRICING_VERSION, "ERROR",            "UpstreamError: fx-rates returned 503, no quote available",            trace_id, error_kind="UpstreamError", upstream="fx-rates"))        docs.append(log(            started + timedelta(milliseconds=4), "checkout-api", checkout_version, "ERROR",            "UpstreamError: pricing-api returned 500, checkout aborted",            trace_id, error_kind="UpstreamError", upstream="pricing-api"))    elif i % 10 == 0:        docs.append(log(started, "checkout-api", checkout_version, "INFO",                        "checkout completed", trace_id))# --- search-api, the coincidence: its own traffic, its own traces -----------for i in range(SEARCH_REQUESTS):    started = WINDOW_START + timedelta(seconds=random.uniform(0, window_seconds))    trace_id = hexid(32)    timed_out = started >= SEARCH_BREAKS_AT and random.random() < SEARCH_TIMEOUT_RATE    docs.append(span(started, "search-api", SEARCH_VERSION, "GET /search",                     504 if timed_out else 200, trace_id,                     random.uniform(2000, 3000) if timed_out else random.uniform(30, 90)))    if timed_out:        docs.append(log(started + timedelta(milliseconds=1), "search-api", SEARCH_VERSION,                        "ERROR",                        "SearchTimeout: query exceeded 2000ms during catalog reindex",                        trace_id, error_kind="SearchTimeout"))    elif i % 10 == 0:        docs.append(log(started, "search-api", SEARCH_VERSION, "INFO",                        "search served", trace_id))print(f"{len(docs):,} documents to index")

In [ ]:
ok, errors = helpers.bulk(es, docs, chunk_size=2000, raise_on_error=False, stats_only=False)print(f"indexed {ok:,}")if errors:    print(f"{len(errors)} failures, first one:")    print(json.dumps(errors[0], indent=2)[:1200])es.indices.refresh(index=f"{LOGS},{TRACES}")

## 4. Check the on-call viewExpect four unhealthy services, with `search-api` on top at about 36%.

In [ ]:
def esql(query, **params):    body = {"query": query, "format": "txt"}    if params:        body["params"] = [{k: v} for k, v in params.items()]    return es.esql.query(**body).bodyprint(esql(f"""FROM {TRACES}| WHERE kind == "Server"| EVAL failed = CASE(attributes.http.status_code >= 500, 1, 0)| STATS failures = SUM(failed), requests = COUNT(*)    BY service = resource.attributes.service.name| EVAL failure_rate_pct = ROUND(100.0 * failures / requests, 1)| SORT failure_rate_pct DESC"""))

## 5. Check the two refutationsExpect two failure shapes with no overlap, and two `checkout-api` versions failing at about thesame rate. Without that split the agent has nothing to rule out.

In [ ]:
START = ts(WINDOW_START)END = ts(WINDOW_END)print(esql(f"""FROM {LOGS}| WHERE @timestamp >= TO_DATETIME(?start) AND @timestamp <= TO_DATETIME(?end)  AND severity_text == "ERROR" AND trace_id IS NOT NULL| STATS services = VALUES(resource.attributes.service.name) BY trace_id| EVAL failure_shape = MV_CONCAT(MV_SORT(services), " + ")| STATS traces = COUNT(*) BY failure_shape| SORT traces DESC| LIMIT 20""", start=START, end=END))print(esql(f"""FROM {TRACES}| WHERE @timestamp >= TO_DATETIME(?start) AND @timestamp <= TO_DATETIME(?end)  AND resource.attributes.service.name == ?service AND kind == "Server"| EVAL failed = CASE(attributes.http.status_code >= 500, 1, 0)| STATS failures = SUM(failed), requests = COUNT(*)    BY version = resource.attributes.service.version| EVAL failure_rate_pct = ROUND(100.0 * failures / requests, 1)| SORT version""", start=START, end=END, service="checkout-api"))

## 6. Register the toolsEach description states the decision the tool supports, which is what the model reads to decidewhen to call it.

In [ ]:
TOOLS = [    {        "id": "rca_failure_shapes",        "type": "esql",        "description": (            "Use this to decide whether two services that started failing at the same time are "            "one cascading failure or two unrelated ones. Groups error logs by trace and returns "            "how many traces each combination of erroring services appears in. Services in one "            "cascade share trace ids; two separate failures have zero overlap."        ),        "tags": ["rca", "evidence"],        "configuration": {            "query": (                f"FROM {LOGS}\n"                "| WHERE @timestamp >= TO_DATETIME(?start) AND @timestamp <= TO_DATETIME(?end)\n"                "  AND severity_text == \"ERROR\" AND trace_id IS NOT NULL\n"                "| STATS services = VALUES(resource.attributes.service.name) BY trace_id\n"                "| EVAL failure_shape = MV_CONCAT(MV_SORT(services), \" + \")\n"                "| STATS traces = COUNT(*) BY failure_shape\n"                "| SORT traces DESC\n"                "| LIMIT 20"            ),            "params": {                "start": {"type": "text", "description": "Window start, ISO 8601", "optional": False},                "end": {"type": "text", "description": "Window end, ISO 8601", "optional": False},            },        },    },    {        "id": "rca_version_split",        "type": "esql",        "description": (            "Use this to decide whether a recent deploy caused an incident. Returns the request "            "failure rate of one service split by service.version, from server spans. A release "            "that caused the failures makes its version fail at a materially higher rate than the "            "version it replaced. Requires both versions to have served traffic in the window."        ),        "tags": ["rca", "evidence"],        "configuration": {            "query": (                f"FROM {TRACES}\n"                "| WHERE @timestamp >= TO_DATETIME(?start) AND @timestamp <= TO_DATETIME(?end)\n"                "  AND resource.attributes.service.name == ?service AND kind == \"Server\"\n"                "| EVAL failed = CASE(attributes.http.status_code >= 500, 1, 0)\n"                "| STATS failures = SUM(failed), requests = COUNT(*)\n"                "    BY version = resource.attributes.service.version\n"                "| EVAL failure_rate_pct = ROUND(100.0 * failures / requests, 1)\n"                "| SORT version"            ),            "params": {                "start": {"type": "text", "description": "Window start, ISO 8601", "optional": False},                "end": {"type": "text", "description": "Window end, ISO 8601", "optional": False},                "service": {"type": "text", "description": "Service name", "optional": False},            },        },    },    {        "id": "rca_evidence_sample",        "type": "esql",        "description": (            "Use this to get citable evidence for a claim. Returns raw error log records for one "            "service with their Elasticsearch document ids and trace ids, so a finding can point "            "at a specific record a reader can open in Discover."        ),        "tags": ["rca", "evidence"],        "configuration": {            "query": (                f"FROM {LOGS} METADATA _id, _index\n"                "| WHERE @timestamp >= TO_DATETIME(?start) AND @timestamp <= TO_DATETIME(?end)\n"                "  AND severity_text == \"ERROR\" AND resource.attributes.service.name == ?service\n"                "| KEEP @timestamp, _id, _index, trace_id,\n"                "       attributes.error.kind, attributes.upstream.service, body.text\n"                "| SORT @timestamp DESC\n"                "| LIMIT 5"            ),            "params": {                "start": {"type": "text", "description": "Window start, ISO 8601", "optional": False},                "end": {"type": "text", "description": "Window end, ISO 8601", "optional": False},                "service": {"type": "text", "description": "Service name", "optional": False},            },        },    },]for tool in TOOLS:    requests.delete(f"{BASE}/api/agent_builder/tools/{tool['id']}", headers=headers, timeout=30)    r = requests.post(f"{BASE}/api/agent_builder/tools", headers=headers, json=tool, timeout=30)    if not r.ok:        raise RuntimeError(f"{tool['id']}: {r.status_code} {r.text[:500]}")    print(tool["id"], "ok")

## 7. Wire the agent

In [ ]:
INSTRUCTIONS = """You are an SRE assistant. Help the user find the root cause of production incidents.1. Scope. Establish the affected service, the failure window, and the size of the symptom before   you name any cause. State the window as an explicit ISO 8601 range and reuse that same range in   every tool call.2. Enumerate. Write down at least three candidate causes before you test any of them. Include the   candidate a human on call would reach for first, such as a recent deploy or another service that   started failing at the same minute.3. Refute. For each candidate, state the observation that would prove it wrong, then run the query   that produces that observation. A candidate is dismissed when the refuting observation is   present, not when a different candidate looks better.4. Cite. Every number you report must name the tool that returned it. Every claim about a root   cause must carry at least one trace_id and at least one document _id. A claim with no citation   is not a finding, it is a guess.5. Report. Put anything the available data cannot settle in gaps_found rather than resolving it   with reasoning.Two rules:- Do not rank services by error volume and call the loudest one the cause.- Do not treat time correlation as causation. Two services that start failing in the same minute   belong to the same incident only if they appear in the same traces."""AGENT = {    "id": "rca_evidence_agent",    "type": "chat",    "name": "Evidence-first RCA",    "description": "Root cause analysis that refutes candidates and cites the record behind every claim.",    "labels": ["rca", "observability"],    "configuration": {        "instructions": INSTRUCTIONS,        "tools": [            {                "tool_ids": [                    "rca_failure_shapes",                    "rca_version_split",                    "rca_evidence_sample",                    "observability.get_logs",                    "observability.get_traces",                    "observability.get_apm_correlations",                    "observability.get_log_change_points",                    "platform.streams.investigation_progress_report",                ]            }        ],    },}requests.delete(f"{BASE}/api/agent_builder/agents/{AGENT['id']}", headers=headers, timeout=30)r = requests.post(f"{BASE}/api/agent_builder/agents", headers=headers, json=AGENT, timeout=30)if not r.ok:    raise RuntimeError(f"{AGENT['id']}: {r.status_code} {r.text[:600]}")print(AGENT["id"], "ok")

## 8. Run the investigationTakes about a minute. Tool calls print as they happen.A 403 here means the API key cannot reach Agent Builder chat. Open the agent in Kibana under**Agent Builder > Agents > Evidence-first RCA** and paste the same question there.

In [ ]:
QUESTION = f"""checkout-api started returning HTTP 500s to customers today. Investigate thewindow {START} to {END} and tell me the root cause. Context you have from the deploy log:checkout-api release {NEW_VERSION} rolled out at 09:55 UTC, and the on-call channel alsoreported search-api timeouts starting at 09:55 UTC."""print(QUESTION)# converse is an internal Kibana route, so it needs the internal-origin header on top of# kbn-xsrf. Without it Kibana answers 403 even when the API key is allowed to create agents.chat_headers = {**headers, "x-elastic-internal-origin": "Kibana"}def converse(agent_id, message):    """Read the SSE stream from Agent Builder and print tool calls as they happen."""    answer = []    with requests.post(        f"{BASE}/api/agent_builder/converse/async",        headers=chat_headers,        json={"agent_id": agent_id, "input": message},        stream=True,        timeout=600,    ) as r:        if not r.ok:            raise RuntimeError(f"{r.status_code} {r.text[:500]}")        event = ""        for raw in r.iter_lines(decode_unicode=True):            if raw is None or raw.startswith(":"):                continue            if raw.startswith("event: "):                event = raw[7:].strip()            elif raw.startswith("data: "):                data = json.loads(raw[6:]).get("data", {})                if event == "tool_call":                    print(f"  -> {data.get('tool_id')} {json.dumps(data.get('params', {}))[:160]}")                elif event == "message_complete":                    answer.append(data.get("message_content") or data.get("message") or "")    return "\n".join(part for part in answer if part)report = converse(AGENT["id"], QUESTION)print("\n" + report)

## 9. Verify a citationPaste a `trace_id` from the report into `TRACE_ID`, or leave it as is to pick one. `fx-rates`errs first with no upstream, because it is the origin.

In [ ]:
TRACE_ID = "paste a trace_id from the report above"if TRACE_ID.startswith("paste"):    row = es.esql.query(query=f"""        FROM {LOGS}        | WHERE severity_text == "ERROR"          AND resource.attributes.service.name == "fx-rates"        | KEEP trace_id        | LIMIT 1    """).body    TRACE_ID = row["values"][0][0]    print(f"using {TRACE_ID}")print(esql(f"""FROM {LOGS}| WHERE trace_id == "{TRACE_ID}"| KEEP @timestamp, resource.attributes.service.name,       attributes.upstream.service, body.text| SORT @timestamp ASC"""))

## 10. Clean upDeletes the agent, the tools, and only the documents written above. The `delete_by_query` isscoped to the four demo service names.

In [ ]:
requests.delete(f"{BASE}/api/agent_builder/agents/{AGENT['id']}", headers=headers, timeout=30)for tool in TOOLS:    requests.delete(f"{BASE}/api/agent_builder/tools/{tool['id']}", headers=headers, timeout=30)for index in (LOGS, TRACES):    result = es.delete_by_query(        index=index,        query={"terms": {"resource.attributes.service.name": DEMO_SERVICES}},        refresh=True,        conflicts="proceed",    )    print(f"{index}: deleted {result['deleted']:,}")print("cleaned up")